# Marker gene heatmap for single cell data.
- 24 single cell samples plotted.
- according to the perct_1 annotation for clarity.
- assumes standard_scale = 'var' instead of z-score. (check the last part of the script for explanation).

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
adata = sc.read_h5ad(f'{ACC_DATA_ROOT}/outputs/h5ad_files/inprogress_3.h5ad')

In [ ]:
adata

In [ ]:
set(list(adata.obs['prect_1']))

In [ ]:
marker_genes_dict = {
  "Actively Dividing cells": ["MKI67", "TOP2A", "UBE2C"],

  "B cells": ["MS4A1", "CD79A", "CD74"],

  "Basal cells": ["KRT5", "KRT14", "TP63"],

  "Endothelial cells": ["PECAM1", "VWF", "KDR"],

  "Epithelial cells": ["EPCAM", "KRT19", "KRT8"],

  "Fibroblast cells": ["COL1A1", "DCN", "LUM"],

  "Macrophage cells": ["LST1", "C1QA", "TYROBP"],

  "Mast cells": ["KIT", "CPA3", "MS4A2"],

  "Mural cells": ["RGS5", "PDGFRB", "CSPG4"],

  "Muscle cells": ["ACTA1", "TNNT3", "CKM"],

  "Plasma cells": ["MZB1", "IGKC", "JCHAIN"],

  "T cells": ["TRAC", "CD3D", "CD3E"],
}

In [ ]:
# considering a fixed number of cells from every cluster to plot the marker gene heatmap.
n_top_cells = 50

selected_indices = []

for cell_type, genes in marker_genes_dict.items():
    cells = adata.obs_names[adata.obs['prect_1'] == cell_type]

    if len(genes) == 0 or len(cells) == 0:
        continue

    valid_genes = [g for g in genes if g in adata.var_names]
    if not valid_genes:
        continue

    expr = adata[cells, valid_genes].X.toarray() if hasattr(adata.X, 'toarray') else adata[cells, valid_genes].X

    scores = expr.sum(axis=1)

    top_idx = np.argsort(scores)[-n_top_cells:]

    selected_cells = cells[top_idx]
    selected_indices.extend(selected_cells)

adata_top = adata[selected_indices]

# display order of cell types

desired_order = [
    "Actively Dividing cells",
    "B cells",
    "Basal cells",
    "Endothelial cells",
    "Epithelial cells",
    "Fibroblast cells",
    "Macrophage cells",
    "Mast cells",
    "Mural cells",
    "Muscle cells",
    "Plasma cells",
    "T cells",
]

adata.obs['prect_1'] = adata.obs['prect_1'].astype('category')
adata.obs['prect_1'] = adata.obs['prect_1'].cat.set_categories(desired_order, ordered=True)

# Plot heatmap using Scanpy's inbuilt heatmap function.

sc.pl.heatmap(
    adata_top,
    marker_genes_dict,
    groupby='prect_1',
    cmap="YlOrRd",
    dendrogram=False,
    show_gene_labels=True,
    swap_axes=True,
    standard_scale='var',  
    show=False,
    figsize=(12, 15)
)

axes = plt.gcf().get_axes()
for ax in axes:
    ax.tick_params(axis='both', labelsize=14)
    # Also increase group/category labels if present
    ax.set_ylabel(ax.get_ylabel(), fontsize=14)
    ax.set_xlabel(ax.get_xlabel(), fontsize=14)

plt.tight_layout()

plt.close()

# standard_scale = 'var'
- Standardizes the expression of each gene across the groups plotted, so expression patterns can be compared across groups.
- Colors do not represent absolute expression values.